In [1]:
from datasets import load_dataset
import pandas as pd

/Users/administrador/Desktop/2026_Hack4Edu/Hack4Edu_2026/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Math afrimgsm

In [85]:
data_english_math = load_dataset('masakhane/afrimgsm', 'eng') 
data_yor_math = load_dataset('masakhane/afrimgsm', "yor") 

In [86]:
data_english_math, data_yor_math

(DatasetDict({
     train: Dataset({
         features: ['question', 'answer', 'answer_number', 'equation_solution'],
         num_rows: 8
     })
     test: Dataset({
         features: ['question', 'answer', 'answer_number', 'equation_solution'],
         num_rows: 250
     })
 }),
 DatasetDict({
     train: Dataset({
         features: ['question', 'answer', 'answer_number', 'equation_solution'],
         num_rows: 8
     })
     test: Dataset({
         features: ['question', 'answer', 'answer_number', 'equation_solution'],
         num_rows: 250
     })
 }))

In [87]:
data_yor_math["test"]["equation_solution"]

Column([None, None, None, None, None, ...])

In [88]:
data_yor_math["train"]["answer"][0]

'Ìdáhùn lẹ́sẹsẹ: Roger bẹ̀rẹ̀ pẹ̀lú bọ́ọ́lù 5. Agolo 2 bọ́ọ́lù aláfajọ̀gbá kọ̀ọ̀kan 3 jẹ́ bọ́ọ́lù aláfajọ̀gbá 6. 5 + 6 = 11. Ìdáhùn náà ni 11.'

In [89]:
rag_rows = []

In [90]:


for split in ["train", "test"]:
    number_rows = len(data_english_math[split]["question"])
    
    for index in range(number_rows):
        q_yor = data_yor_math[split]["question"][index]
        q_english = data_english_math[split]["question"][index]
        
        ans_yor = data_yor_math[split]["answer"][index]
        
        ans_num = data_yor_math[split]["answer_number"][index]
        eq_english = data_yor_math[split]["equation_solution"][index]
        
        
        context = f"{q_yor} | {q_english}"
        
        
        if ans_yor is not None and eq_english is not None:
            answer = (
                f"[Verified Translation]: {q_english}\n"
                f"[Yorùbá Step-by-Step]: {ans_yor}\n"
                f"[Math Equation]: {eq_english}\n"
                f"[Exact Final Answer]: {ans_num}"
            )
        
        else:
            answer = (
                f"[Verified Parallel Math Problem (Yorùbá -> English)]:\n"
                f"- Yorùbá Text: {q_yor}\n"
                f"- Exact English Meaning: {q_english}\n"
                f"- Verified Numerical Result: {ans_num}"
            )
        
        rag_rows.append({
            "language": "yor",
            "source_dataset": f"afrimgsm_{split}",
            "search_context": context,
            "pedagogical_answer": answer
        })



### Multiple question for general intelligence (afrimmlu)

In [91]:
data_multiple_questions_eng = load_dataset('masakhane/afrimmlu', 'eng')
data_multiple_questions_yor = load_dataset('masakhane/afrimmlu', 'yor')

In [92]:
data_multiple_questions_eng 

DatasetDict({
    validation: Dataset({
        features: ['subject', 'question', 'choices', 'answer'],
        num_rows: 83
    })
    dev: Dataset({
        features: ['subject', 'question', 'choices', 'answer'],
        num_rows: 25
    })
    test: Dataset({
        features: ['subject', 'question', 'choices', 'answer'],
        num_rows: 500
    })
})

### Looking for Nan's


In [ ]:
print(data_multiple_questions_eng["validation"]["subject"].count(None) , data_multiple_questions_eng["dev"]["subject"].count(None) , data_multiple_questions_eng["test"]["subject"].count(None))
print(data_multiple_questions_eng["validation"]["question"].count(None) , data_multiple_questions_eng["dev"]["question"].count(None) , data_multiple_questions_eng["test"]["question"].count(None))
print(data_multiple_questions_eng["validation"]["choices"].count(None) , data_multiple_questions_eng["dev"]["choices"].count(None) , data_multiple_questions_eng["test"]["choices"].count(None))print(data_multiple_questions_eng["validation"]["answer"].count(None) , data_multiple_questions_eng["dev"]["answer"].count(None) , data_multiple_questions_eng["test"]["answer"].count(None))

0 0 0
0 0 0
0 0 0
0 0 0


In [ ]:
for split in ["validation", "dev", "test"]:
    number_rows = len(data_multiple_questions_eng[split]["question"])
    for index in range(number_rows):
        q_yor = data_multiple_questions_yor[split]["question"][index]
        q_english = data_multiple_questions_eng[split]["question"][index]
        
        eng_sub = data_multiple_questions_eng[split]["subject"][index]
        
        yor_answer = data_multiple_questions_yor[split]["answer"][index]
        eng_anwser = data_multiple_questions_eng[split]["answer"][index]
        
        assert data_multiple_questions_eng[split]["subject"][index] == data_multiple_questions_yor[split]["subject"][index], "Must be the same"
        assert data_multiple_questions_eng[split]["answer"][index] == data_multiple_questions_yor[split]["answer"][index], "Must be the same"
        
                
        context = f"{q_yor} | {q_english} | {eng_sub}"
        
        answer =  (
                f"[Subject: {eng_sub} | Academic Bilingue Reference]:\n"
                f"- Yorùbá Question: {q_yor}\n"
                f"- English Question: {q_english}\n"
                f"- Verified Answer (Yorùbá): {yor_answer}\n"
                f"- Verified Answer (English): {eng_anwser}"
            )
        
        rag_rows.append({
            "language": "yor",
                "source_dataset": f"afrimmlu_{split}",
                "search_context": context,
                "pedagogical_answer": answer
            })

### Lecture comprehesion

In [102]:
data_reading_comphresion_yor = load_dataset('facebook/belebele', "yor_Latn")
data_reading_comphresion_english = load_dataset('facebook/belebele', "eng_Latn")

Generating test split: 100%|██████████| 900/900 [00:00<00:00, 89711.34 examples/s]


In [111]:
data_reading_comphresion_yor["test"]["correct_answer_num"]

Column(['1', '1', '2', '2', '3', ...])

In [104]:
data_reading_comphresion_english

DatasetDict({
    test: Dataset({
        features: ['link', 'question_number', 'flores_passage', 'question', 'mc_answer1', 'mc_answer2', 'mc_answer3', 'mc_answer4', 'correct_answer_num', 'dialect', 'ds'],
        num_rows: 900
    })
})

In [ ]:
for split in ["test"]:
    number_rows = len(data_reading_comphresion_english[split]["question"])
    for index in range(number_rows)

900
